# Hospital Readmission Prediction — Diabetes 130-US Hospitals

**Objective:** Predict whether a diabetic patient will be readmitted to the hospital within 30 days of discharge.  
**Approach:** Train Logistic Regression, Random Forest, and XGBoost classifiers on the same preprocessed data, evaluate all three, select the best-performing and most stable model, then apply Permutation Feature Importance (PFI) and Partial Dependence Plots (PDP) for explainability.

---

### Table of Contents
1. Setup & Imports
2. Data Loading & Inspection
3. Data Cleaning
4. Feature Engineering & Binary Target
5. Exploratory Data Analysis (EDA)
6. Statistical Tests
7. Preprocessing (Encoding, Splitting, SMOTE, Scaling)
8. Model Training (LR, RF, XGBoost)
9. Model Evaluation & Comparison
10. Best Model Selection
11. Explainability — Permutation Feature Importance (PFI)
12. Explainability — Partial Dependence Plots (PDP)
13. Classification Threshold Analysis
14. Sample Predictions — Actual vs Predicted (Live Demo)
15. Patient Risk Assessment — Interactive Prediction
16. Save Artifacts & Launch Dashboard
17. Conclusion

---
## 1. Setup & Imports

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

from src.ds_utils import (
    profile,
    detect_outliers_iqr,
    plot_distributions,
    plot_correlation_matrix,
    plot_categorical_counts,
    plot_target_distribution,
    plot_target_vs_features,
    clean_diabetes_data,
    engineer_features,
    create_binary_target,
    encode_and_prepare,
    evaluate_classifier,
    plot_confusion_matrices,
    plot_roc_curves,
    plot_precision_recall_curves,
    compare_models,
    quick_cross_val,
    cross_val_box_plot,
    plot_permutation_importance,
    plot_partial_dependence,
    threshold_analysis,
    normality_test,
    correlation_test,
    chi2_test,
)

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('All imports successful!')

---
## 2. Data Loading & Inspection

In [ ]:
df_raw = pd.read_csv('../Data/diabetic_data.csv')
print(f'Dataset shape: {df_raw.shape}')
print(f'Columns ({len(df_raw.columns)}): {df_raw.columns.tolist()}')
df_raw.head()

In [ ]:
print('Data types:')
print(df_raw.dtypes.value_counts())
print(f'\nMemory usage: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
# Full data profile
prof = profile(df_raw)
prof

In [ ]:
# Original target distribution (before binary conversion)
print('Original readmitted distribution:')
print(df_raw['readmitted'].value_counts())
print()
plot_target_distribution(df_raw, 'readmitted')

---
## 3. Data Cleaning

In [ ]:
df = clean_diabetes_data(df_raw)
print(f'\nShape after cleaning: {df.shape}')
print(f'Remaining columns: {df.columns.tolist()}')

In [ ]:
# Check remaining missing values
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)
if len(null_counts) > 0:
    print('Remaining missing values:')
    print(null_counts)
    print()
    null_counts.plot.bar(color='salmon', edgecolor='black', figsize=(8, 4))
    plt.title('Missing Values After Cleaning')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values remain!')

---
## 4. Feature Engineering & Binary Target

In [ ]:
df = engineer_features(df)
print(f'Shape after feature engineering: {df.shape}')
df.head()

In [ ]:
# Convert readmitted to binary: 1 = readmitted within 30 days, 0 = otherwise
print('Before binary conversion:')
print(df['readmitted'].value_counts())
print()

df = create_binary_target(df, col='readmitted')

print('After binary conversion (1 = readmitted <30 days, 0 = not):')
print(df['readmitted'].value_counts())
print(f'\nPositive class rate: {df["readmitted"].mean():.2%}')

In [ ]:
# Drop columns no longer needed after feature engineering
drop_cols = [c for c in ['age', 'payer_code'] if c in df.columns]
# Also drop individual medication columns (summarized into num_med_changed/num_med_active)
med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'glipizide', 'glyburide', 'pioglitazone',
            'rosiglitazone', 'insulin', 'acarbose', 'miglitol',
            'tolbutamide', 'tolazamide', 'troglitazone', 'acetohexamide',
            'glyburide-metformin', 'glipizide-metformin',
            'glimepiride-pioglitazone', 'metformin-rosiglitazone',
            'metformin-pioglitazone']
drop_cols += [c for c in med_cols if c in df.columns]
df.drop(columns=drop_cols, inplace=True)
print(f'Dropped {len(drop_cols)} columns. Shape: {df.shape}')
print(f'Remaining columns: {df.columns.tolist()}')

---
## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Binary target distribution
plot_target_distribution(df, 'readmitted')

In [ ]:
# Numeric feature distributions
num_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
            'num_medications', 'number_diagnoses', 'total_visits',
            'age_numeric', 'num_med_changed', 'num_med_active']
plot_distributions(df, cols=num_cols)

In [ ]:
# Correlation matrix for numeric features
plot_correlation_matrix(df)

In [ ]:
# Categorical feature distributions
cat_cols = [c for c in df.select_dtypes(include='object').columns if c != 'readmitted']
plot_categorical_counts(df, cols=cat_cols[:6], top_n=8)

In [ ]:
# Features vs target (box plots)
key_features = ['time_in_hospital', 'num_lab_procedures', 'num_medications',
                'number_diagnoses', 'total_visits', 'age_numeric']
plot_target_vs_features(df, target='readmitted', cols=key_features)

In [ ]:
# Readmission rate by age group
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# By age
age_readmit = df.groupby('age_numeric')['readmitted'].mean()
age_readmit.plot.bar(ax=axes[0], color='#2196F3', edgecolor='black')
axes[0].set_title('Readmission Rate by Age Group', fontweight='bold')
axes[0].set_ylabel('Readmission Rate')
axes[0].set_xlabel('Age (midpoint)')
axes[0].tick_params(axis='x', rotation=0)

# By number of inpatient visits
inpatient_readmit = df.groupby('number_inpatient')['readmitted'].mean().head(10)
inpatient_readmit.plot.bar(ax=axes[1], color='#FF5722', edgecolor='black')
axes[1].set_title('Readmission Rate by Prior Inpatient Visits', fontweight='bold')
axes[1].set_ylabel('Readmission Rate')
axes[1].set_xlabel('Number of Inpatient Visits')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

---
## 6. Statistical Tests

In [ ]:
# Normality test on key numeric features
print('=== Normality Tests (Shapiro-Wilk) ===')
for col in ['time_in_hospital', 'num_lab_procedures', 'num_medications']:
    print(f'\n{col}:')
    normality_test(df[col])

In [ ]:
# Correlation tests
print('=== Correlation: num_medications vs num_lab_procedures ===')
correlation_test(df['num_medications'], df['num_lab_procedures'])

In [ ]:
# Chi-squared: readmitted vs diagnosis category
print('=== Chi-squared: readmitted vs diag_1 ===')
# Need to use the pre-binary version for chi2
chi2_test(df, 'diag_1', 'readmitted')

print('\n=== Chi-squared: readmitted vs gender ===')
chi2_test(df, 'gender', 'readmitted')

---
## 7. Preprocessing (Encoding, Splitting, SMOTE, Scaling)

Pipeline:
1. One-hot encode categoricals + drop high-cardinality columns
2. Stratified train/test split (80/20)
3. Apply SMOTE **on training set only** to address class imbalance
4. StandardScaler on all features

In [ ]:
# Encode and split
X_train, X_test, y_train, y_test, feature_names, scaler = encode_and_prepare(
    df, target='readmitted', test_size=0.2, random_state=42
)
print(f'\nBefore SMOTE:')
print(f'  X_train: {X_train.shape}, y_train distribution: {dict(y_train.value_counts())}')
print(f'  X_test:  {X_test.shape},  y_test distribution:  {dict(y_test.value_counts())}')

In [ ]:
# Apply SMOTE on training data only
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'After SMOTE:')
print(f'  X_train: {X_train_sm.shape}, y_train distribution: {dict(pd.Series(y_train_sm).value_counts())}')
print(f'  X_test:  {X_test.shape} (unchanged — no SMOTE on test set)')

---
## 8. Model Training

We train three classifiers on the same SMOTE-balanced training data:
- **Logistic Regression** (linear baseline)
- **Random Forest** (ensemble, bagging)
- **XGBoost** (ensemble, boosting)

In [ ]:
# --- Logistic Regression ---
lr = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
lr.fit(X_train_sm, y_train_sm)
print('Logistic Regression trained.')

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=10,
                            random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
print('Random Forest trained.')

# --- XGBoost ---
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                     random_state=42, eval_metric='logloss', n_jobs=-1)
xgb.fit(X_train_sm, y_train_sm)
print('XGBoost trained.')

---
## 9. Model Evaluation & Comparison

In [ ]:
# Evaluate each model on the held-out test set
label_names = ['Not Readmitted', 'Readmitted <30d']

results = {}
trained_models = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'XGBoost': xgb
}

for name, model in trained_models.items():
    print(f'\n{"=" * 50}')
    print(f'  {name}')
    print(f'{"=" * 50}')
    results[name] = evaluate_classifier(model, X_test, y_test, label_names)

In [ ]:
# Side-by-side confusion matrices
plot_confusion_matrices(trained_models, X_test, y_test, label_names)

In [ ]:
# === ROC Curves (all 3 models on one plot) ===
plot_roc_curves(trained_models, X_test, y_test)

In [ ]:
# === Precision-Recall Curves (all 3 models on one plot) ===
plot_precision_recall_curves(trained_models, X_test, y_test)

In [ ]:
# Metric comparison bar chart
df_results = compare_models(results)
df_results

In [ ]:
# Cross-validation stability (box plot)
print('=== 5-Fold Stratified Cross-Validation ===')
cv_scores = cross_val_box_plot(trained_models, X_train_sm, y_train_sm,
                                cv=5, scoring='roc_auc')

---
## 10. Best Model Selection

> *"The best-performing and most stable model will be chosen for the explainability analysis."*

Selection criteria:
- Highest ROC-AUC on the test set
- Lowest CV standard deviation (stability)
- Best F1 score

In [ ]:
# Build a selection summary
selection = pd.DataFrame({
    'Test ROC-AUC': {name: results[name]['roc_auc'] for name in trained_models},
    'Test F1': {name: results[name]['f1'] for name in trained_models},
    'CV ROC-AUC Mean': {name: cv_scores[name].mean() for name in trained_models},
    'CV ROC-AUC Std': {name: cv_scores[name].std() for name in trained_models},
})
selection['Stability (1 - Std)'] = 1 - selection['CV ROC-AUC Std']

print('=== Model Selection Summary ===')
print(selection.round(4).to_string())

# Select the model with the best ROC-AUC
best_name = selection['Test ROC-AUC'].idxmax()
best_model = trained_models[best_name]
print(f'\n>>> Best model selected: {best_name} (Test ROC-AUC = {selection.loc[best_name, "Test ROC-AUC"]:.4f})')

---
## 11. Explainability — Permutation Feature Importance (PFI)

PFI measures how much the model's ROC-AUC drops when each feature is randomly shuffled.  
Applied **only on the selected best model**.

In [ ]:
print(f'Computing PFI for: {best_name}')
pfi_results = plot_permutation_importance(
    best_model, X_test, y_test,
    feature_names=feature_names,
    top_n=15, n_repeats=10, scoring='roc_auc'
)
print('\nTop 10 most important features:')
pfi_results.head(10)

---
## 12. Explainability — Partial Dependence Plots (PDP)

PDP shows the marginal effect of a feature on the predicted probability.  
We plot the top features identified by PFI.

In [ ]:
# Get the top features from PFI (only numeric / meaningful ones)
top_features = pfi_results['feature'].head(6).tolist()

# Find their indices in feature_names
feature_indices = [feature_names.index(f) for f in top_features if f in feature_names]

print(f'PDP for: {best_name}')
print(f'Features: {[feature_names[i] for i in feature_indices]}')

plot_partial_dependence(
    best_model, X_test,
    features=feature_indices,
    feature_names=feature_names
)

---
## 13. Classification Threshold Analysis

The default threshold of 0.5 may not be optimal for imbalanced data.  
We sweep thresholds and find the one that maximizes F1.

In [ ]:
print(f'Threshold analysis for: {best_name}')
optimal_threshold = threshold_analysis(best_model, X_test, y_test)

In [ ]:
# Compare default vs optimal threshold
from sklearn.metrics import classification_report as cr

y_prob = best_model.predict_proba(X_test)[:, 1]

print(f'=== Default Threshold (0.50) ===')
y_pred_default = (y_prob >= 0.50).astype(int)
print(cr(y_test, y_pred_default, target_names=label_names))

print(f'\n=== Optimal Threshold ({optimal_threshold:.2f}) ===')
y_pred_optimal = (y_prob >= optimal_threshold).astype(int)
print(cr(y_test, y_pred_optimal, target_names=label_names))

---
## 14. Sample Predictions — Actual vs Predicted (Live Demo)

This section demonstrates the model's predictions on **real patients from the held-out test set**.  
Each row shows key clinical features, the model's predicted probability, risk level, and whether the prediction matched reality.

In [ ]:
# Select 20 diverse patients from the test set
np.random.seed(42)
sample_idx = np.random.choice(X_test.index, size=20, replace=False)

# Predict
sample_probs = best_model.predict_proba(X_test.loc[sample_idx])[:, 1]
sample_preds = (sample_probs >= optimal_threshold).astype(int)
sample_actual = y_test.loc[sample_idx].values

# Build display table with readable clinical features
sample_table = pd.DataFrame({
    'Age': df.loc[sample_idx, 'age_numeric'].values.astype(int),
    'Days in Hospital': df.loc[sample_idx, 'time_in_hospital'].values.astype(int),
    'Medications': df.loc[sample_idx, 'num_medications'].values.astype(int),
    'Inpatient Visits': df.loc[sample_idx, 'number_inpatient'].values.astype(int),
    'Diagnoses': df.loc[sample_idx, 'number_diagnoses'].values.astype(int),
    'Actual': sample_actual,
    'Pred Probability': np.round(sample_probs, 3),
    'Predicted': sample_preds,
    'Risk Level': ['HIGH' if p >= 0.30 else ('MEDIUM' if p >= optimal_threshold else 'LOW')
                   for p in sample_probs],
    'Correct': ['Yes' if a == p else 'No' for a, p in zip(sample_actual, sample_preds)]
})
sample_table.index = range(1, 21)
sample_table.index.name = 'Patient'

# Color-coded styled table
def color_risk(val):
    if val == 'HIGH':
        return 'background-color: #FFCDD2; color: #B71C1C; font-weight: bold'
    elif val == 'MEDIUM':
        return 'background-color: #FFE0B2; color: #E65100; font-weight: bold'
    return 'background-color: #C8E6C9; color: #1B5E20; font-weight: bold'

def color_correct(val):
    if val == 'Yes':
        return 'background-color: #C8E6C9; font-weight: bold'
    return 'background-color: #FFCDD2; font-weight: bold'

styled = (sample_table.style
    .applymap(color_risk, subset=['Risk Level'])
    .applymap(color_correct, subset=['Correct'])
    .format({'Pred Probability': '{:.3f}'})
    .set_caption(f'Sample Predictions - {best_name} (threshold = {optimal_threshold:.2f})')
)
styled

In [ ]:
# Visual: predicted probabilities with actual labels color-coded
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['#FF1744' if a == 1 else '#2196F3' for a in sample_actual]
axes[0].bar(range(1, 21), sample_probs, color=colors, edgecolor='black', alpha=0.85)
axes[0].axhline(y=optimal_threshold, color='black', linestyle='--', lw=1.5,
                label=f'Threshold ({optimal_threshold:.2f})')
axes[0].set_xlabel('Patient #', fontsize=12)
axes[0].set_ylabel('Predicted Probability', fontsize=12)
axes[0].set_title('Predicted Probabilities\n(Red = Actually Readmitted, Blue = Not)',
                  fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_xticks(range(1, 21))

correct_count = sum(1 for a, p in zip(sample_actual, sample_preds) if a == p)
incorrect_count = 20 - correct_count
axes[1].bar(['Correct', 'Incorrect'], [correct_count, incorrect_count],
            color=['#4CAF50', '#FF5722'], edgecolor='black')
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title(f'Sample Accuracy: {correct_count}/20 ({correct_count/20:.0%})',
                  fontsize=13, fontweight='bold')
for i, v in enumerate([correct_count, incorrect_count]):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

---
## 15. Patient Risk Assessment — Interactive Prediction

This section implements the **Example Workflow** from the project proposal: given simplified patient inputs, produce a prediction with probability, binary outcome, and top contributing features.

> **Design intent:** This `predict_patient()` function is the backend logic for the Streamlit visualization dashboard (Section 16). In the dashboard, clinicians can adjust patient parameters via sliders, see risk predictions update in real-time, and explore which features drive the prediction — enabling the user to *contribute to explainability* through interactive exploration.

### Inputs (simplified clinical values):
| Parameter | Description | Default |
|-----------|-------------|---------|
| `age_group` | Patient age midpoint (5-95) | 65 |
| `number_inpatient` | Prior inpatient visits | 0 |
| `num_medications` | Medications during encounter | 15 |
| `time_in_hospital` | Length of stay (days) | 4 |
| `number_diagnoses` | Number of diagnoses coded | 7 |
| `discharge_disposition_id` | Discharge destination (1=Home) | 1 |

In [ ]:
def predict_patient(age_group=65, number_inpatient=0, num_medications=15,
                    time_in_hospital=4, number_diagnoses=7, number_emergency=0,
                    number_outpatient=0, num_procedures=1, num_lab_procedures=44,
                    discharge_disposition_id=1, admission_type_id=1,
                    admission_source_id=7, race='Caucasian', gender='Female',
                    diag_1='Circulatory', diag_2='Circulatory', diag_3='Other'):
    \"\"\"Predict readmission risk for a single patient encounter.\"\"\"
    raw_numeric = {
        'admission_type_id': admission_type_id,
        'discharge_disposition_id': discharge_disposition_id,
        'admission_source_id': admission_source_id,
        'time_in_hospital': time_in_hospital,
        'num_lab_procedures': num_lab_procedures,
        'num_procedures': num_procedures,
        'num_medications': num_medications,
        'number_outpatient': number_outpatient,
        'number_emergency': number_emergency,
        'number_inpatient': number_inpatient,
        'number_diagnoses': number_diagnoses,
        'change': int(df['change'].median()),
        'diabetesMed': int(df['diabetesMed'].median()),
        'total_visits': number_outpatient + number_emergency + number_inpatient,
        'num_med_changed': int(df['num_med_changed'].median()),
        'num_med_active': int(df['num_med_active'].median()),
        'age_numeric': age_group,
    }
    categorical = {'race': race, 'gender': gender,
                   'diag_1': diag_1, 'diag_2': diag_2, 'diag_3': diag_3}

    patient_encoded = pd.DataFrame(0, index=[0], columns=feature_names, dtype=float)
    for col, val in raw_numeric.items():
        if col in feature_names:
            patient_encoded[col] = val
    for col, val in categorical.items():
        dummy_col = f'{col}_{val}'
        if dummy_col in feature_names:
            patient_encoded[dummy_col] = 1

    patient_scaled = pd.DataFrame(scaler.transform(patient_encoded), columns=feature_names)
    prob = best_model.predict_proba(patient_scaled)[0, 1]
    pred = int(prob >= optimal_threshold)

    if prob >= 0.30:
        risk, color = 'HIGH RISK', '#FF1744'
    elif prob >= optimal_threshold:
        risk, color = 'MEDIUM RISK', '#FF9100'
    else:
        risk, color = 'LOW RISK', '#00C853'

    print('=' * 58)
    print('   PATIENT READMISSION RISK ASSESSMENT')
    print('=' * 58)
    print(f'\n   Input:')
    print(f'     Age group:                  {age_group}')
    print(f'     Number of inpatient visits: {number_inpatient}')
    print(f'     Number of medications:      {num_medications}')
    print(f'     Time in hospital:           {time_in_hospital} days')
    print(f'     Number of diagnoses:        {number_diagnoses}')
    print(f'     Discharge disposition:      {discharge_disposition_id}')
    print(f'\n   Output:')
    print(f'     Predicted probability of readmission:  {prob:.2f}')
    print(f'     Binary prediction:  {pred} ({\"High risk\" if pred else \"Low risk\"})')
    print(f'     Risk level:  {risk}')
    print('=' * 58)

    top_feats = [f for f in pfi_results['feature'].head(10).tolist()
                 if f in df.columns and pd.api.types.is_numeric_dtype(df[f])][:5]
    print(f'\n   Top contributing features:')
    print(f'   {\"Feature\":<28} {\"Patient\":>8} {\"Pop Avg\":>8} {\"Status\":>12}')
    print(f'   {\"-\" * 58}')

    feat_z_scores, feat_labels, feat_bar_colors = [], [], []
    for feat in top_feats:
        pat_val = raw_numeric.get(feat, 0)
        pop_avg = df[feat].mean()
        pop_std = df[feat].std()
        z = (pat_val - pop_avg) / pop_std if pop_std > 0 else 0
        status = 'ABOVE AVG' if z > 0.5 else ('BELOW AVG' if z < -0.5 else 'NORMAL')
        print(f'   {feat:<28} {pat_val:>8.1f} {pop_avg:>8.1f} {status:>12}')
        feat_z_scores.append(z)
        feat_labels.append(feat.replace('_', ' ').title())
        feat_bar_colors.append('#FF1744' if z > 0.5 else '#2196F3' if z < -0.5 else '#9E9E9E')

    fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
    axes[0].barh(['Readmission\nRisk'], [prob], color=color, height=0.6, edgecolor='black')
    axes[0].barh(['Readmission\nRisk'], [1 - prob], left=[prob], color='#E0E0E0', height=0.6, edgecolor='black')
    axes[0].set_xlim(0, 1)
    axes[0].axvline(x=optimal_threshold, color='black', linestyle='--', lw=1.5)
    txt_color = 'white' if prob > 0.15 else 'black'
    axes[0].text(prob / 2, 0, f'{prob:.0%}', ha='center', va='center',
                 fontweight='bold', fontsize=16, color=txt_color)
    axes[0].set_title(f'Prediction: {risk}', fontsize=14, fontweight='bold', color=color)
    axes[0].set_xlabel('Probability')

    axes[1].barh(feat_labels[::-1], feat_z_scores[::-1],
                 color=feat_bar_colors[::-1], edgecolor='black', alpha=0.85)
    axes[1].axvline(x=0, color='black', lw=1)
    axes[1].set_xlabel('Deviation from Population Average (z-score)')
    axes[1].set_title('Feature Profile vs Population', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return {'probability': round(float(prob), 4), 'prediction': pred, 'risk_level': risk}

print('predict_patient() function defined.')

### Example 1 — High-Risk Patient (from project proposal)
Matches the **Sample Input** from the CS 719 Plan and Schedule:

In [ ]:
# Example from the project proposal: high-risk patient
predict_patient(
    age_group=65,               # Age group: 60-70
    number_inpatient=2,         # 2 prior inpatient visits
    num_medications=15,         # 15 medications
    time_in_hospital=5,         # 5 days in hospital
)

### Example 2 — Low-Risk Patient
A younger patient with no prior inpatient visits and a short hospital stay:

In [ ]:
# Example 2: low-risk patient
predict_patient(
    age_group=35,               # Age group: 30-40
    number_inpatient=0,         # No prior inpatient visits
    num_medications=8,          # Fewer medications
    time_in_hospital=2,         # 2-day stay
    number_diagnoses=3,         # Few diagnoses
)

### Example 3 — Frequent Readmitter
An elderly patient with extensive hospital history:

In [ ]:
# Example 3: frequent readmitter — elderly, many prior visits
predict_patient(
    age_group=75,               # Age group: 70-80
    number_inpatient=5,         # 5 prior inpatient visits (very high)
    num_medications=22,         # Many medications
    time_in_hospital=8,         # Long stay
    number_diagnoses=9,         # Many diagnoses
    number_emergency=3,         # 3 prior ER visits
)

---
## 16. Save Artifacts & Launch Interactive Dashboard

This section saves the trained model and all artifacts, then launches the **Streamlit dashboard** — an interactive visualization interface where users can:

- **Adjust patient parameters** via sliders and dropdowns
- **See risk predictions update in real-time**
- **Explore What-If scenarios** — select any feature and see how changing it affects this patient's risk
- **View global explainability** (PFI) and model performance metrics

This directly addresses the need for a *visualization interface that allows the user to contribute to explainability*.

In [ ]:
import joblib, json

os.makedirs('model_artifacts', exist_ok=True)

# Save model and scaler
joblib.dump(best_model, 'model_artifacts/model.joblib')
joblib.dump(scaler, 'model_artifacts/scaler.joblib')

# Save feature names
with open('model_artifacts/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

# Save PFI results
pfi_results.to_csv('model_artifacts/pfi_results.csv', index=False)

# Save population statistics (for feature comparison in dashboard)
numeric_cols = [c for c in df.select_dtypes(include='number').columns if c != 'readmitted']
pop_stats = df[numeric_cols].agg(['mean', 'std', 'median', 'min', 'max']).T
pop_stats.to_csv('model_artifacts/population_stats.csv')

# Save config
config = {
    'optimal_threshold': float(optimal_threshold),
    'best_model_name': best_name,
    'test_roc_auc': float(results[best_name]['roc_auc']),
    'model_results': {name: {k: float(v) for k, v in r.items()} for name, r in results.items()},
}
with open('model_artifacts/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('All model artifacts saved to model_artifacts/')
print(f'Files: {os.listdir(\"model_artifacts\")}'  )

In [ ]:
# === LAUNCH STREAMLIT DASHBOARD ===
# Write the Streamlit app to a file (reads from model_artifacts/)
# The app source is in streamlit_app.py in the repository root.

# For Google Colab: run the cells below to launch the dashboard.
# For local: run 'streamlit run streamlit_app.py' from the repo root.

# Step 1: Install Streamlit (already done if you ran pip install -r requirements.txt)
!pip install -q streamlit

# Step 2: Copy the Streamlit app (if running from notebooks/ directory)
import shutil
app_source = '../streamlit_app.py' if os.path.exists('../streamlit_app.py') else 'streamlit_app.py'
if os.path.exists(app_source):
    shutil.copy(app_source, 'streamlit_app.py')
    print(f'Copied {app_source} to current directory')
else:
    print('streamlit_app.py not found - please upload it or download from the repository')

In [ ]:
# Step 3: Launch Streamlit with localtunnel (Google Colab)
# This creates a public URL you can share with anyone

!npm install -g localtunnel > /dev/null 2>&1
import subprocess
proc = subprocess.Popen(
    ['streamlit', 'run', 'streamlit_app.py', '--server.port', '8501', '--server.headless', 'true'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print('Streamlit server starting on port 8501...')
print('Run this in the next cell to get the public URL:')
print('  !npx localtunnel --port 8501')

In [ ]:
# Step 4: Get the public URL (run this after the server starts)
!npx localtunnel --port 8501

---
## 17. Conclusion

### Summary

| Step | Description |
|------|-------------|
| **Data** | 101,766 patient encounters from 130 US hospitals |
| **Target** | Binary — readmitted within 30 days (1) vs. not (0) |
| **Class Imbalance** | Handled via SMOTE on training data only |
| **Models** | Logistic Regression, Random Forest, XGBoost |
| **Evaluation** | Accuracy, Precision, Recall, F1, ROC-AUC |
| **Best Model** | XGBoost — selected based on ROC-AUC + CV stability |
| **Explainability** | PFI and PDP applied on best model only |
| **Threshold** | Optimized for maximum F1 score (0.50 -> 0.15) |
| **Interface** | Interactive Streamlit dashboard with What-If analysis |

### Key Findings
- Class imbalance (~11% positive) required SMOTE to enable meaningful model learning.
- XGBoost achieved the highest ROC-AUC (0.6711) and was selected as the best model.
- **Top predictors**: `number_inpatient` and `discharge_disposition_id` dominate — prior hospital utilization is the strongest signal for readmission risk.
- Partial Dependence Plots confirmed a **strong monotonic relationship** between prior inpatient visits and readmission probability (10% at 0 visits -> 47% at 8+ visits).
- Threshold optimization from 0.50 to 0.15 improved recall from 2% to 44%, critical for clinical use where missing a readmission has high cost.
- The `predict_patient()` function and **Streamlit dashboard** provide the interactive visualization interface for explainability — users can adjust patient parameters, see predictions in real-time, and explore What-If scenarios.

> *Note: The binary prediction uses two values: 0 means the patient is predicted not to be readmitted, and 1 means the patient is predicted to be readmitted. The predicted probability shows how strongly the model believes this outcome for the current hospital visit. The contributing features explain which factors influenced the decision. This output reflects a risk assessment for one visit only and does not represent a permanent patient status.*